#### OC Transpo Optimization

2026-03-09

In [3]:
#imports
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from pathlib import Path


# -----------------------------
# file / sheet settings
# -----------------------------
DATA_FILE = Path("data/ridership_jan_2023_subset.csv")

In [4]:
# -----------------------------
# loading
# -----------------------------
def load_ridership_data(file_path=DATA_FILE):
    """
    Load the cleaned OC Transpo weekday ridership subset from CSV.
    """
    df = pd.read_csv(file_path)
    return df

In [5]:
ridership = load_ridership_data()
ridership

,Route,Early AM,AM Peak,Midday,PM Peak,Evening,Night
0,5,23,292,566,303,162,73
1,6,177,1724,3513,2781,1064,448
2,7,110,1649,3533,2555,1075,550
3,9,44,383,368,328,105,58
4,10,42,879,2275,1291,443,229


#### 1. Demand Model
Estimate new ridership from change in service frequency. Use elasticity to define a diminishing return, as would be expected if you started cranking up the service frequency. 

$$
x_{new} = x_{old} \left(\frac{f_{new}}{f_{old}}\right)^{elasticity}
$$

where  
- $x_{old}$ = baseline ridership  
- $f_{old}$ = current frequency  
- $f_{new}$ = proposed frequency  
- $\epsilon$ = demand elasticity (≈0.5)

In [6]:
def demand_from_frequency(f_new, x_old, f_old, elasticity=0.5):
    f_new = np.maximum(f_new, 1e-6)   # avoid divide-by-zero
    f_old = np.maximum(f_old, 1e-6)

    return x_old * (f_new / f_old) ** elasticity
